In [1]:
import pandas as pd
import numpy as np

In [49]:
from LoraCollisionLSTM import LoraCollisionLSTM
#import load_and_prepare_data from functions
from functions import load_and_prepare_data
import os
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
import math
import torch.optim as optim
import itertools
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# BI-LSTM baselines

In [26]:
df = pd.read_csv('baselines_bilstm_results.csv')
df.head()

,combination,seed,model,accuracy,precision,recall,f1_score,auc_roc
0,0,0,comb_0_seed_0_BiLSTM_32_1_loss_0.3288,0.907951,0.842589,0.977389,0.904997,0.914428
1,0,1,comb_0_seed_1_BiLSTM_32_1_loss_0.3304,0.914688,0.843862,0.975879,0.905081,0.923418
2,0,2,comb_0_seed_2_BiLSTM_32_1_loss_0.3438,0.905942,0.839611,0.980429,0.904572,0.912131
3,0,3,comb_0_seed_3_BiLSTM_32_1_loss_0.3475,0.909893,0.838935,0.981011,0.904427,0.918119
4,0,4,comb_0_seed_4_BiLSTM_32_1_loss_0.3457,0.909370,0.840984,0.984855,0.907251,0.916223


In [8]:
for combination in df['combination'].unique():
    subset = df[df['combination'] == combination]

    print(f"Combination: {combination+1}")
    #print(subset[['seed', 'test_mse']])
    #print("\n")

Combination: 1
Combination: 2
Combination: 3
Combination: 4
Combination: 5
Combination: 6
Combination: 7
Combination: 8
Combination: 9
Combination: 10
Combination: 11
Combination: 12


In [ ]:
subset = df[df['combination'] == 0]

subset

,combination,seed,model,accuracy,precision,recall,f1_score,auc_roc
0,0,0,comb_0_seed_0_BiLSTM_32_1_loss_0.3288,0.907951,0.842589,0.977389,0.904997,0.914428
1,0,1,comb_0_seed_1_BiLSTM_32_1_loss_0.3304,0.914688,0.843862,0.975879,0.905081,0.923418
2,0,2,comb_0_seed_2_BiLSTM_32_1_loss_0.3438,0.905942,0.839611,0.980429,0.904572,0.912131
3,0,3,comb_0_seed_3_BiLSTM_32_1_loss_0.3475,0.909893,0.838935,0.981011,0.904427,0.918119
4,0,4,comb_0_seed_4_BiLSTM_32_1_loss_0.3457,0.909370,0.840984,0.984855,0.907251,0.916223
5,0,5,comb_0_seed_5_BiLSTM_32_1_loss_0.3628,0.913843,0.845113,0.980765,0.907900,0.921752
6,0,6,comb_0_seed_6_BiLSTM_32_1_loss_0.4213,0.899184,0.833979,0.967887,0.895957,0.905601
7,0,7,comb_0_seed_7_BiLSTM_32_1_loss_0.3292,0.901803,0.833543,0.979074,0.900467,0.908354
8,0,8,comb_0_seed_8_BiLSTM_32_1_loss_0.4147,0.906613,0.833425,0.984974,0.902885,0.914917
9,0,9,comb_0_seed_9_BiLSTM_32_1_loss_0.3354,0.912602,0.847193,0.973398,0.905921,0.919852


In [27]:
summary = df.groupby('combination')['f1_score'].agg(['mean', 'std']).reset_index()
summary['penalized_score'] = summary['mean'] - summary['std']
summary_sorted = summary.sort_values(by='penalized_score', ascending=False).reset_index(drop=True)

summary_sorted

,combination,mean,std,penalized_score
0,4,0.906237,0.004048,0.902189
1,3,0.905988,0.003930,0.902058
2,5,0.906401,0.004856,0.901545
3,1,0.905701,0.004520,0.901181
4,0,0.903946,0.003504,0.900441
5,2,0.905506,0.005250,0.900255
6,10,0.904350,0.004333,0.900017
7,11,0.903540,0.003859,0.899681
8,9,0.903402,0.004403,0.898999
9,8,0.902368,0.005343,0.897024


In [28]:
summary = df.groupby('combination')[['accuracy', 'precision', 'recall', 'f1_score','auc_roc']].agg(['mean', 'std']).reset_index()
#summary['penalized_score'] = summary['mean'] - summary['std']
summary

combination  accuracy           precision              recall            \
                    mean       std      mean       std      mean       std   
0            0  0.908189  0.005027  0.839923  0.004976  0.978566  0.005212   
1            1  0.909751  0.006154  0.840849  0.005339  0.981420  0.006070   
2            2  0.909529  0.006731  0.840478  0.008806  0.981505  0.004303   
3            3  0.909876  0.005468  0.839808  0.006281  0.983543  0.005789   
4            4  0.910424  0.005467  0.842518  0.007264  0.980466  0.007056   
5            5  0.909799  0.006557  0.836655  0.008199  0.988892  0.003808   
6            6  0.905198  0.004512  0.846110  0.007468  0.959945  0.010460   
7            7  0.904318  0.006878  0.844657  0.008164  0.960012  0.013301   
8            8  0.907440  0.006730  0.844665  0.009839  0.968682  0.009224   
9            9  0.908170  0.005801  0.843542  0.008427  0.972581  0.011268   
10          10  0.909032  0.005966  0.844200  0.008325  0.973850  0.008460   
11          11  0.908179  0.005528  0.842685  0.006695  0.973945  0.007138   

    f1_score             auc_roc            
        mean       std      mean       std  
0   0.903946  0.003504  0.915479  0.005676  
1   0.905701  0.004520  0.917241  0.007168  
2   0.905506  0.005250  0.916998  0.007064  
3   0.905988  0.003930  0.917509  0.006063  
4   0.906237  0.004048  0.917653  0.005874  
5   0.906401  0.004856  0.917995  0.007010  
6   0.899373  0.003508  0.910828  0.004762  
7   0.898569  0.005888  0.910137  0.007706  
8   0.902368  0.005343  0.913765  0.006905  
9   0.903402  0.004403  0.914816  0.006297  
10  0.904350  0.004333  0.915738  0.006414  
11  0.903540  0.003859  0.915027  0.006193

In [16]:
# Setting params    
import itertools

param_grid = {
    'hidden_size' : [32, 64], #128                # Dimension of the mode
    'num_layers'  : [1, 2, 3],                    # Number of encoder layers
    'dropout'     : [0.1, 0.2], # , 0.2                # doprout
    'batch_size'  : [32]                          # Batch Size
}


# Generate the combination of parameters
keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

for i, config in enumerate(combinations):
    print(f"---> Processing Combination {i}/{len(combinations)} ---")
    print(f"Configuration: {config} - i: {i}")    

---> Processing Combination 0/12 ---
Configuration: {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.1, 'batch_size': 32} - i: 0
---> Processing Combination 1/12 ---
Configuration: {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.2, 'batch_size': 32} - i: 1
---> Processing Combination 2/12 ---
Configuration: {'hidden_size': 32, 'num_layers': 2, 'dropout': 0.1, 'batch_size': 32} - i: 2
---> Processing Combination 3/12 ---
Configuration: {'hidden_size': 32, 'num_layers': 2, 'dropout': 0.2, 'batch_size': 32} - i: 3
---> Processing Combination 4/12 ---
Configuration: {'hidden_size': 32, 'num_layers': 3, 'dropout': 0.1, 'batch_size': 32} - i: 4
---> Processing Combination 5/12 ---
Configuration: {'hidden_size': 32, 'num_layers': 3, 'dropout': 0.2, 'batch_size': 32} - i: 5
---> Processing Combination 6/12 ---
Configuration: {'hidden_size': 64, 'num_layers': 1, 'dropout': 0.1, 'batch_size': 32} - i: 6
---> Processing Combination 7/12 ---
Configuration: {'hidden_size': 64, 'num_layers': 1, '

In [45]:
# Select the best configuration based on the analysis
# Select the 4th Configuration 

best_params = {'hidden_size': 32, 'num_layers': 3, 'dropout': 0.1, 'batch_size': 32}
best_params

{'hidden_size': 32, 'num_layers': 3, 'dropout': 0.1, 'batch_size': 32}

In [47]:
def train_test_lstm(config, train_loader, val_loader, device, epochs=20):
    logs = {}
    logs['train_loss_log']  = []    
    logs['train_acc_log']   = []
    logs['val_loss_log']    = []
    logs['val_acc_log']     = []

    # Create model with configuration
    model_lstm = LoraCollisionLSTM(        
        num_features = 9, # 8 reales + 1 Delta_T (este se adiciona en otras funciones)
        hidden_size  = config['hidden_size'], #64, 
        num_layers   = config['num_layers'], #2, 
        dropout      = config['dropout'] #0.1
    ).to(device)

    criterion = nn.BCELoss()
    #optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)
    
    #best_test_acc = 0.0   
    best_val_acc = 0.0 
    #best_val_loss = float('inf')
    avg_val_loss = 0.0
    for epoch in range(epochs):
        # Train
        model_lstm.train()
        train_loss      = 0.0
        correct_train   = 0
        total_train     = 0

        #for X_b, y_b in train_loader:
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            # Forward pass
            predictions = model_lstm(batch_X) # Usa internamente el squeeze(-1)

            # Clean gradient 
            optimizer.zero_grad()

            # Calculate error loss
            loss = criterion(predictions, batch_y)

            # Backpropagation
            loss.backward()

            # Update weights
            optimizer.step()

            # loss.item - The average of the batch losses will give you an estimate of the “epoch loss” during training. Returns the value of this tensor as a standard Python number 
            train_loss += loss.item() * batch_X.size(0)
            
            # Convert to 0 - 1 the vector batch
            predictions_class = (predictions >= 0.5).float()
            
            correct_train += (predictions_class == batch_y).sum().item()
            total_train   += batch_y.size(0)

            
        
        avg_train_loss  = train_loss / total_train
        train_acc       = correct_train / total_train
        
        logs['train_loss_log'].append(avg_train_loss)
        logs['train_acc_log'].append(train_acc)

        # Validation
        model_lstm.eval()        
        val_loss     = 0.0  
        correct_val = 0
        total_val   = 0
        
        y_true = []
        y_pred = []
        with torch.no_grad():
            #for X_b, y_b in val_loader:
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                
                # Predict
                predictions = model_lstm(batch_X)
                loss        = criterion(predictions, batch_y)
                val_loss   += loss.item() * batch_X.size(0)

                # Calculate correct or not 
                predictions_class = (predictions >= 0.5).float()
                correct_val      += (predictions_class == batch_y).sum().item()
                total_val        += batch_y.size(0)

                # Other method to calculate metrics
                y_pred.extend(predictions_class.cpu().numpy())  
                y_true.extend(batch_y.cpu().numpy())
                
        
        avg_val_loss = val_loss / total_val
        val_acc      = correct_val / total_val

        
        logs['val_loss_log'].append(avg_val_loss)
        logs['val_acc_log'].append(val_acc)
        
        # Print metrics
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        auc = roc_auc_score(y_true, y_pred)
        

        print(f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {avg_train_loss:.4f} - Acc: {train_acc*100:.1f}% | "
        f"Val Loss: {avg_val_loss:.4f} - Acc: {val_acc*100:.1f}% | " 
        f"Val Acc: {acc:.4f} - Val Prec: {prec*100:.1f}% | "
        f"Val Rec: {rec:.4f} - Val F1: {f1*100:.1f}% | "
        f"Val AUC-ROC: {auc:.4f} "
        )
    return avg_val_loss, model_lstm, logs, y_true, y_pred

In [42]:
base_path = 'data/repetitions/'
num_seeds = 10
features = ['latDev', 'longDev', 'elevSat', 'loraTP', 'loraSF', 'doppler', 'alt', 'raan']

device = "cpu"
print(f"Using : {device}")    

# Read all csv files and create datasets for each seed
dict_train_dataset = {} 
dict_test_dataset  = {}
dict_val_dataset   = {}
for seed in range(num_seeds):
    print(f"---> Reading files for Seed {seed+1}/{num_seeds} ---")

    # Construcción dinámica de nombres de archivos
    path_train = os.path.join(base_path, f'seed_{seed}_final_train_data_transmission.csv')
    path_test  = os.path.join(base_path, f'seed_{seed}_final_test_data_transmission.csv')
    path_val   = os.path.join(base_path, f'seed_{seed}_final_val_data_transmission.csv')

    X_train, y_train, X_test, y_test, X_val  , y_val, scaler = load_and_prepare_data(seq_length = 16, global_path_train=path_train, global_path_test=path_test, global_path_val=path_val)

    # Crteata Datasets
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset  = TensorDataset(X_test, y_test)
    val_dataset   = TensorDataset(X_val, y_val)

    # Store Datasets in dictionaries
    dict_train_dataset[seed] = train_dataset
    dict_test_dataset[seed] = test_dataset
    dict_val_dataset[seed] = val_dataset

Using : cpu
---> Reading files for Seed 1/10 ---
Loading dataset...


/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


---> Reading files for Seed 2/10 ---
Loading dataset...


/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


---> Reading files for Seed 3/10 ---
Loading dataset...


...

/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


---> Reading files for Seed 9/10 ---
Loading dataset...


/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


---> Reading files for Seed 10/10 ---
Loading dataset...


/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [50]:
best_val_loss_seed = float('inf')
best_logs_seed = None
best_config_seed = None
results_list_seed = []

config = best_params
print(f"Selected Configuration for Seed Analysis: {config}")
for seed in range(num_seeds):
    print(f"-------> Processing Seed {seed+1}/{num_seeds} ---")
    # Get DataLoaders from dictionaries
    train_dataset = dict_train_dataset[seed]
    test_dataset  = dict_test_dataset[seed]
    val_dataset   = dict_val_dataset[seed]
    
    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    test_loader  = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)
    val_loader   = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)     
            
    # Training
    final_loss, model_lstm, logs, y_true, y_pred = train_test_lstm(config, train_loader, val_loader, device, epochs=20)
    
    print(f"-> Best Loss: {final_loss:.4f}")                    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_pred)

    metrics = {        
        "seed": seed,
        "model": 'analyzed_' + 'seed_' + str(seed) + '_BiLSTM_' + str(config['hidden_size']) + '_' + str(config['num_layers']) + '_loss_' + str(round(final_loss, 4)) ,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "auc_roc": auc
    }
    results_list_seed.append(metrics)


    if final_loss < best_val_loss_seed:
        print(f"New best model found for the seed {seed} with val_loss: {final_loss:.4f} (previous best: {best_val_loss_seed:.4f}). Saving model...")
        name_model = 'analyzed_' + 'seed_' + str(seed) + '_BiLSTM_' + str(config['hidden_size']) + '_' + str(config['num_layers']) + '_loss_' + str(round(final_loss, 4)) + '.pth'
        best_val_loss_seed = final_loss
        best_logs_seed = logs
        best_config_seed = config
        torch.save(model_lstm.state_dict(), 'models/analyzed/bilstm/' + name_model)
        print("Model saved !!!")

    # Delete model to free memory
    del model_lstm        
    del train_loader
    del test_loader
    del val_loader
    del train_dataset
    del test_dataset
    del val_dataset


    import gc
    gc.collect()

Selected Configuration for Seed Analysis: {'hidden_size': 32, 'num_layers': 3, 'dropout': 0.1, 'batch_size': 32}
-------> Processing Seed 1/10 ---
Epoch 01/20 | Train Loss: 0.2924 - Acc: 86.8% | Val Loss: 0.2306 - Acc: 90.4% | Val Acc: 0.9042 - Val Prec: 82.7% | Val Rec: 0.9947 - Val F1: 90.3% | Val AUC-ROC: 0.9127 
Epoch 02/20 | Train Loss: 0.2529 - Acc: 88.7% | Val Loss: 0.2326 - Acc: 90.3% | Val Acc: 0.9030 - Val Prec: 82.7% | Val Rec: 0.9913 - Val F1: 90.2% | Val AUC-ROC: 0.9112 
Epoch 03/20 | Train Loss: 0.2467 - Acc: 89.0% | Val Loss: 0.2249 - Acc: 90.6% | Val Acc: 0.9063 - Val Prec: 83.1% | Val Rec: 0.9932 - Val F1: 90.5% | Val AUC-ROC: 0.9144 
Epoch 04/20 | Train Loss: 0.2423 - Acc: 89.2% | Val Loss: 0.2671 - Acc: 90.5% | Val Acc: 0.9048 - Val Prec: 82.6% | Val Rec: 0.9977 - Val F1: 90.4% | Val AUC-ROC: 0.9135 
Epoch 05/20 | Train Loss: 0.2376 - Acc: 89.4% | Val Loss: 0.2381 - Acc: 90.8% | Val Acc: 0.9083 - Val Prec: 83.4% | Val Rec: 0.9934 - Val F1: 90.7% | Val AUC-ROC: 0.9163

KeyboardInterrupt: 

In [51]:
best_params

{'hidden_size': 32, 'num_layers': 3, 'dropout': 0.1, 'batch_size': 32}

In [55]:
print("\n--- EVALUACIÓN FINAL EN TEST SET ---")
device = "cpu"
# Instanciar modelo con la configuración actual

final_model = LoraCollisionLSTM(        
    num_features = 9, # 8 reales + 1 Delta_T (este se adiciona en otras funciones)
    hidden_size  = best_params['hidden_size'], #64, 
    num_layers   = best_params['num_layers'], #2, 
    dropout      = best_params['dropout'] #0.1
).to(device)

#name_to_load = 'Transformer_32_4_loss_0.0882.pth'
base_path = 'models/analyzed/bilstm/'
best_name_model = base_path + 'analyzed_seed_0_BiLSTM_32_3_loss_0.305.pth'
train_dataset = dict_train_dataset[0]
test_dataset  = dict_test_dataset[0]
val_dataset   = dict_val_dataset[0]

test_loader  = DataLoader(test_dataset, batch_size=best_params['batch_size'], shuffle=False)

# 2. Cargas los pesos guardados (.pth)
final_model.load_state_dict(torch.load(best_name_model))
final_model.eval() # Modo evaluación 

criterion = nn.BCELoss()

#optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)

# 3. Evaluación final (Una sola pasada)
test_loss = 0.0
correct_test = 0
total_test = 0
y_true, y_pred = [], []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # Solo predecimos y calculamos error, NO hacemos .backward() ni .step()
        predictions = final_model(batch_X)
        loss = criterion(predictions, batch_y)
        
        # Estadísticas
        test_loss += loss.item() * batch_X.size(0)
        preds_clase = (predictions >= 0.5).float()
        correct_test += (preds_clase == batch_y).sum().item()
        total_test += batch_y.size(0)

        # Other method to calculate metrics
        y_pred.extend(preds_clase.cpu().numpy())  
        y_true.extend(batch_y.cpu().numpy())


 # --- REPORTE DEL EPOCH ---
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_true, y_pred)
print(f"FINAL REPORT"        
        f"Test Acc: {acc:.4f} - Val Prec: {prec*100:.1f}% | "
        f"Test Rec: {rec:.4f} - Val F1: {f1*100:.1f}% | "
        f"Test AUC-ROC: {auc:.4f}"
        )


print(f"F1 Final en Test: {f1_score(y_true, y_pred):.4f}")


--- EVALUACIÓN FINAL EN TEST SET ---
FINAL REPORTTest Acc: 0.8737 - Val Prec: 80.9% | Test Rec: 0.9664 - Val F1: 88.1% | Test AUC-ROC: 0.8769
F1 Final en Test: 0.8806


# Other ML baselines

In [56]:
df = pd.read_csv('baselines_ml_results.csv')
df.head()

,seed,model,accuracy,precision,recall,f1_score,auc_roc
0,0,Logistic Regression,0.818044,0.829115,0.781392,0.804546,0.816585
1,0,Random Forest,0.909684,0.875023,0.946777,0.909487,0.911161
2,0,MLP,0.882003,0.824508,0.957619,0.886092,0.885014
3,1,Logistic Regression,0.824995,0.836674,0.782921,0.808905,0.822847
4,1,Random Forest,0.895507,0.850224,0.945730,0.895438,0.898071


In [57]:
summary = df.groupby('model')[['accuracy', 'precision', 'recall', 'f1_score','auc_roc']].agg(['mean', 'std']).reset_index()
#summary['penalized_score'] = summary['mean'] - summary['std']
summary

model  accuracy           precision              recall  \
                            mean       std      mean       std      mean   
0  Logistic Regression  0.819664  0.003946  0.829105  0.008658  0.779669   
1                  MLP  0.877525  0.004319  0.828296  0.008276  0.935159   
2        Random Forest  0.898339  0.009450  0.857036  0.013423  0.942438   

             f1_score             auc_roc            
        std      mean       std      mean       std  
0  0.011815  0.803538  0.005628  0.817522  0.003861  
1  0.015220  0.878377  0.005304  0.880357  0.004366  
2  0.006987  0.897674  0.009404  0.900584  0.009197

In [60]:
for model in df['model'].unique():
    subset = df[df['model'] == model]

    print(f"Model: {model}")
    for seed in subset['seed'].unique():
        subset_seed = subset[subset['seed'] == seed]
        print(subset_seed.head())
        break
    break

Model: Logistic Regression
   seed                model  accuracy  precision    recall  f1_score  \
0     0  Logistic Regression  0.818044   0.829115  0.781392  0.804546   

    auc_roc  
0  0.816585  


In [73]:
summary = df.groupby(['model', 'seed'])['f1_score'].agg(['mean']).reset_index()

# 1. Encontrar la fila con el 'mean' máximo para cada 'model'
aux = summary.groupby('model')['mean'].idxmax()

# 2. Filtrar el DataFrame original usando esos índices y seleccionar las columnas
results_list_seed_by_model = summary.loc[aux, ['model', 'seed', 'mean']]
results_list_seed_by_model  

,model,seed,mean
6,Logistic Regression,6,0.809749
10,MLP,0,0.886092
20,Random Forest,0,0.909487


In [78]:
base_path = 'data/repetitions/'
features = ['latDev', 'longDev', 'elevSat', 'loraTP', 'loraSF', 'doppler', 'alt', 'raan']

for model in df['model'].unique():
    subset = df[df['model'] == model]

    best_seed_for_model = results_list_seed_by_model[results_list_seed_by_model['model'] == model]['seed'].iloc[0]
    print(f"Best Seed for {model}: {best_seed_for_model}")

    # Read data from the seed
    path_train = os.path.join(base_path, f'seed_{best_seed_for_model}_final_train_data_transmission.csv')
    path_test  = os.path.join(base_path, f'seed_{best_seed_for_model}_final_test_data_transmission.csv')
    path_val   = os.path.join(base_path, f'seed_{best_seed_for_model}_final_val_data_transmission.csv')


    # Lectura de datos
    df_train = pd.read_csv(path_train)
    df_test  = pd.read_csv(path_test)
    df_val   = pd.read_csv(path_val)

    X_train_raw = df_train[features].values
    y_train     = df_train['rcvOk'].values
    X_test_raw  = df_test[features].values
    y_test      = df_test['rcvOk'].values
    X_val_raw   = df_val[features].values
    y_val       = df_val['rcvOk'].values
    # Apply calculation on testing


    scaler = StandardScaler()
    scaler.fit(np.concatenate((X_train_raw, X_test_raw, X_val_raw), axis=0))
    
    
    X_train = scaler.transform(X_train_raw)
    X_test  = scaler.transform(X_test_raw)
    X_val   = scaler.transform(X_val_raw)

    model_ml = None
    if model == "Logistic Regression":        
        model_ml = LogisticRegression(max_iter=1000, random_state=42)
    elif model == "Random Forest":
        model_ml = RandomForestClassifier(n_estimators=100, random_state=42)
    elif model == "MLP":
        model_ml = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42)
    else:
        print(f"Model {model} not recognized. Skipping...")
        break

    model_ml.fit(X_train, y_train)
    y_pred = model_ml.predict(X_test)
    metrics = {
            "seed": best_seed_for_model,
            "model": model,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1_score": f1_score(y_test, y_pred, zero_division=0),
            "auc_roc": roc_auc_score(y_test, y_pred)
        }
    
    print(f"Metrics for {model} with seed {best_seed_for_model}: \n{metrics}")
    

Best Seed for Logistic Regression: 6
Metrics for Logistic Regression with seed 6: 
{'seed': np.int64(6), 'model': 'Logistic Regression', 'accuracy': 0.8212020823473734, 'precision': 0.834717607973422, 'recall': 0.7862311754351653, 'f1_score': 0.8097492194581529, 'auc_roc': 0.8201148540418672}
Best Seed for Random Forest: 0
Metrics for Random Forest with seed 0: 
{'seed': np.int64(0), 'model': 'Random Forest', 'accuracy': 0.90968351440718, 'precision': 0.875022772818364, 'recall': 0.9467770549970431, 'f1_score': 0.9094868396137096, 'auc_roc': 0.9111606610253721}
Best Seed for MLP: 0
Metrics for MLP with seed 0: 
{'seed': np.int64(0), 'model': 'MLP', 'accuracy': 0.8820028341993387, 'precision': 0.8245078071961982, 'recall': 0.957618766016164, 'f1_score': 0.8860921112631099, 'auc_roc': 0.8850140274202736}
